# Dedalus — parametric CAD in a notebook

Dedalus is a standalone package: this notebook uses only `dedalus` and CadQuery.
Every step below is started explicitly; nothing runs automatically.

In [ ]:
from pathlib import Path
import cadquery as cq
from vegeta import dedalus
from vegeta.dedalus import Design, Parameter

RUNS = Path("_runs/dedalus"); RUNS.mkdir(parents=True, exist_ok=True)

## 1. Define a parametric design
Parameters are explicit, typed and range-checked.

In [ ]:
class Bracket(Design):
    parameters = [
        Parameter("length", 80.0, "mm", min=10),
        Parameter("width", 40.0, "mm", min=10),
        Parameter("thickness", 6.0, "mm", min=1),
        Parameter("hole_diameter", 6.5, "mm", min=0.5),
        Parameter("fillet", 4.0, "mm", min=0),
    ]

    def build(self, p):
        body = cq.Workplane("XY").box(p["length"], p["width"], p["thickness"])
        if p["fillet"] > 0:
            body = body.edges("|Z").fillet(p["fillet"])
        dx, dy = p["length"] / 2 - 10, p["width"] / 2 - 10
        return body.faces(">Z").workplane().pushPoints([(dx, dy), (-dx, dy), (dx, -dy), (-dx, -dy)]).hole(p["hole_diameter"])

bracket = Bracket()
bracket.params.table()

## 2. Generate and inspect
The geometry displays with CadQuery's own notebook viewer.

In [ ]:
g = bracket.generate()
g

In [ ]:
g.measure()

In [ ]:
fig = dedalus.plot_views(g)

## 3. Change a parameter — the old geometry is untouched

In [ ]:
g_thick = bracket.generate(thickness=10.0)
print(g.measure()["volume"], g_thick.measure()["volume"])

Invalid values are rejected, never clamped:

In [ ]:
try:
    bracket.generate(thickness=0.2)
except ValueError as e:
    print("rejected:", e)

## 4. Export STEP and STL (explicit)

In [ ]:
res = g_thick.export(RUNS / "bracket_t10")
res

## 5. A small explicit parameter study (no optimisation)

In [ ]:
gs = [bracket.generate(thickness=t) for t in (4.0, 6.0, 8.0, 10.0)]
fig = dedalus.plot_parameter_study(gs, "thickness", "volume")
dedalus.measurements_table(gs)

## 6. Any STEP file can be measured

In [ ]:
dedalus.load_step(res.artifacts["step"]).measure()["volume"]